# 第 35 课：多麦克风与 Delay-and-Sum 波束形成

多个麦克风接收同一声源时存在到达时间差。先对齐目标方向再求和，目标相长、部分噪声相消。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 音频信号前端 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 34 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 到达时间差、delay-and-sum、空间混叠 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：到达时间差、delay-and-sum、空间混叠。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

rng=np.random.default_rng(9);speech,sr=sf.read(ROOT/"data"/"spoken_digits_parts"/"2_jackson_0.wav");speech=speech.astype(np.float32)
def shift(x,n):
    if n>=0:return np.pad(x,(n,0))[:len(x)]
    return np.pad(x[-n:],(0,-n))
mic1=speech+.12*rng.normal(size=len(speech));mic2=shift(speech,3)+.12*rng.normal(size=len(speech))

## 1. 扫描延迟估计相关性

In [ ]:
lags=range(-12,13);scores=[]
for lag in lags:scores.append(np.dot(mic1,shift(mic2,lag)))
best=list(lags)[int(np.argmax(scores))]
plt.stem(list(lags),scores);plt.axvline(best,color="C1");plt.xlabel("Applied delay (samples)");plt.ylabel("Cross-correlation");plt.title("Delay search");plt.show()
print("best alignment delay",best)

## 2. 对齐后求平均

In [ ]:
beam=(mic1+shift(mic2,best))/2
def snr(ref,test):return 10*np.log10(np.mean(ref**2)/(np.mean((test-ref)**2)+1e-12))
print("mic1 SNR",snr(speech,mic1),"beam approximate SNR",snr(speech,beam))

## 3. 真实阵列更复杂

延迟由麦克风几何、声速和方向决定；分数采样延迟需要插值或频域相位旋转。混响、多声源、空间混叠会限制简单 delay-and-sum。MVDR 等方法还会估计空间协方差。

## 本课测试

1. 未对齐直接平均可能发生什么？
2. 麦克风间距越大是否永远越好？
3. 3.5 samples 延迟怎样处理？
4. beamforming 是否等于声源分离？
5. 线上阵列为什么需要校准？

<details><summary>展开参考答案</summary>

1. 目标相消或频谱梳状失真。2. 不是，会产生空间混叠并受设备约束。3. 插值或频域相位。4. 不等于，但可增强特定方向。5. 通道增益、相位、位置和时钟误差会破坏对齐。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 35 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `到达时间差`、`delay-and-sum`、`空间混叠`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**两个通道未对齐就平均**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**估计 delay、对齐并比较 SNR**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把阵列输出接入单通道 NS/VAD**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：到达时间差、delay-and-sum、空间混叠。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 到达时间差、delay-and-sum、空间混叠。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
